In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder

# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---- Load cleaned dataset ----
alloy = pd.read_csv("alloy_data_cleaned.csv")
print(f"Loaded {len(alloy):,} rows, {alloy.shape[1]} columns")

# ---- Define the feature plan ----
# Decisions taken from the data quality findings report.
DROP_COLS = [
    "id", "full_address", "street_address", "unit", "owner_name", "assessor_id",
    "latitude", "longitude", "zip_code",
    "price_per_sqft", "annual_tax", "building_age",
    "last_sale_date", "days_since_sale", "sale_year", "tax_year",
]

# only drop columns that exist (in case alloy_data_cleaned.csv has minor differences)
DROP_COLS = [c for c in DROP_COLS if c in alloy.columns]
alloy = alloy.drop(columns=DROP_COLS)
print(f"After dropping unused columns: {alloy.shape[1]} columns")
print(f"Remaining columns: {alloy.columns.tolist()}")

# ---- Target ----
TARGET = "last_sale_price"
y = alloy[TARGET]

# ---- Two feature sets ----
# Model A includes assessed_value; Model B excludes it.
COMMON_FEATURES = [
    "state", "city", "county",
    "property_type", "owner_occupied",
    "sqft", "lot_size_sqft", "bedrooms", "bathrooms", "year_built",
]
MODEL_A_FEATURES = COMMON_FEATURES + ["assessed_value"]
MODEL_B_FEATURES = COMMON_FEATURES.copy()

# ---- Train/test split ----
# Single split (not CV) for simplicity. Random state pinned for reproducibility.
X_full = alloy[MODEL_A_FEATURES]  # superset; we'll subset per model
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train_full):,} rows | Test: {len(X_test_full):,} rows")

# ---- Target encoding for high-cardinality categoricals ----
# Target encoding replaces a category (e.g. "California") with the mean of the
# target seen in the training set for that category. CRITICAL: fit on training
# data only, then apply to test. Otherwise we leak target information into test.
TARGET_ENCODE_COLS = ["state", "city", "county"]

encoder = TargetEncoder(cols=TARGET_ENCODE_COLS, smoothing=10)
# .copy() to avoid SettingWithCopyWarning
X_train_enc = X_train_full.copy()
X_test_enc = X_test_full.copy()

X_train_enc[TARGET_ENCODE_COLS] = encoder.fit_transform(
    X_train_full[TARGET_ENCODE_COLS], y_train
)
X_test_enc[TARGET_ENCODE_COLS] = encoder.transform(X_test_full[TARGET_ENCODE_COLS])

# ---- Encode the small low-cardinality categoricals as plain category dtype ----
# LightGBM handles these natively; we just need them as 'category'.
for col in ["property_type", "owner_occupied"]:
    X_train_enc[col] = X_train_enc[col].astype("category")
    X_test_enc[col] = X_test_enc[col].astype("category")
    # Make sure test categories match train (LightGBM handles unseen, but cleaner this way)
    X_test_enc[col] = X_test_enc[col].cat.set_categories(X_train_enc[col].cat.categories)

# ---- Two final feature matrices ----
X_train_A = X_train_enc[MODEL_A_FEATURES]
X_test_A  = X_test_enc[MODEL_A_FEATURES]
X_train_B = X_train_enc[MODEL_B_FEATURES]
X_test_B  = X_test_enc[MODEL_B_FEATURES]

print(f"\nModel A features ({len(MODEL_A_FEATURES)}): {MODEL_A_FEATURES}")
print(f"Model B features ({len(MODEL_B_FEATURES)}): {MODEL_B_FEATURES}")
print(f"\nCheck first row of X_train_A:")
print(X_train_A.head(1))

Loaded 1,956 rows, 30 columns
After dropping unused columns: 14 columns
Remaining columns: ['property_id', 'city', 'state', 'county', 'property_type', 'bedrooms', 'bathrooms', 'sqft', 'lot_size_sqft', 'year_built', 'last_sale_price', 'assessed_value', 'owner_occupied', 'coord_matches_state']

Train: 1,564 rows | Test: 392 rows

Model A features (11): ['state', 'city', 'county', 'property_type', 'owner_occupied', 'sqft', 'lot_size_sqft', 'bedrooms', 'bathrooms', 'year_built', 'assessed_value']
Model B features (10): ['state', 'city', 'county', 'property_type', 'owner_occupied', 'sqft', 'lot_size_sqft', 'bedrooms', 'bathrooms', 'year_built']

Sanity check — first row of X_train_A:
         state       city     county property_type owner_occupied  sqft  \
836 550,019.63 728,762.45 728,762.45  Multi Family           True  1308   

     lot_size_sqft  bedrooms  bathrooms  year_built  assessed_value  
836          12923         3       3.00        1931      733,446.68  


In [4]:
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow setup
mlflow.set_tracking_uri("sqlite:///mlflow.db")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

EXPERIMENT_A = "alloytower_avm_modelA_assessment_aware"
EXPERIMENT_B = "alloytower_avm_modelB_fundamentals_only"

mlflow.set_experiment(EXPERIMENT_A)
print(f"Experiment ready: {EXPERIMENT_A}")
mlflow.set_experiment(EXPERIMENT_B)
print(f"Experiment ready: {EXPERIMENT_B}")

# Metrics helper
def compute_metrics(y_true, y_pred):
    """Return MAE, MAPE, R², and RMSE in a dict."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    # MAPE: mean absolute percentage error. Skip rows where y_true is 0.
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

# Residuals plot helper
def save_residuals_plot(y_true, y_pred, out_path, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Predicted vs actual
    axes[0].scatter(y_true, y_pred, alpha=0.3, s=10)
    lim = max(max(y_true), max(y_pred))
    axes[0].plot([0, lim], [0, lim], "r--", linewidth=1)
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predicted")
    axes[0].set_title("Predicted vs Actual")
    axes[0].ticklabel_format(style="plain")

    # Residuals vs predicted
    residuals = np.array(y_true) - np.array(y_pred)
    axes[1].scatter(y_pred, residuals, alpha=0.3, s=10)
    axes[1].axhline(0, color="r", linestyle="--", linewidth=1)
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Residual (actual - predicted)")
    axes[1].set_title("Residuals vs Predicted")
    axes[1].ticklabel_format(style="plain")

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=100, bbox_inches="tight")
    plt.close(fig)

# Baseline 1: predict overall mean
def run_mean_baseline(X_train, X_test, y_train, y_test, experiment_name, model_label):
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=f"{model_label}_mean_baseline"):
        mean_pred = np.full(len(y_test), y_train.mean())
        metrics = compute_metrics(y_test, mean_pred)

        mlflow.log_param("model_type", "mean_baseline")
        mlflow.log_param("features", "none")
        mlflow.log_metrics(metrics)

        print(f"  Mean baseline | MAE: ${metrics['MAE']:>12,.0f} | "
              f"MAPE: {metrics['MAPE']:.1f}% | R²: {metrics['R2']:.3f}")
        return metrics

# Baseline 2: predict per-state mean
def run_state_mean_baseline(X_train, X_test, y_train, y_test, experiment_name, model_label):
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=f"{model_label}_state_mean_baseline"):
        # X_train["state"] is now a target-encoded numeric column — that IS the state mean.
        # So predicting X_test["state"] directly is the state-mean baseline.
        state_pred = X_test["state"].values
        metrics = compute_metrics(y_test, state_pred)

        mlflow.log_param("model_type", "state_mean_baseline")
        mlflow.log_param("features", "state (target encoded)")
        mlflow.log_metrics(metrics)

        print(f"  State-mean    | MAE: ${metrics['MAE']:>12,.0f} | "
              f"MAPE: {metrics['MAPE']:.1f}% | R²: {metrics['R2']:.3f}")
        return metrics

# ---- Linear regression ----
def run_linear_regression(X_train, X_test, y_train, y_test, experiment_name, model_label):
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=f"{model_label}_linear_regression"):
        # LinearRegression doesn't handle category dtype, so drop the small categoricals
        # for this baseline. (Tree models will handle them down the line.)
        cat_cols = [c for c in X_train.columns if str(X_train[c].dtype) == "category"]
        X_train_lr = X_train.drop(columns=cat_cols)
        X_test_lr = X_test.drop(columns=cat_cols)

        # log-transform the target to handle the right skew
        y_train_log = np.log1p(y_train)
        model = LinearRegression()
        model.fit(X_train_lr, y_train_log)
        y_pred_log = model.predict(X_test_lr)
        y_pred = np.expm1(y_pred_log)

        metrics = compute_metrics(y_test, y_pred)

        mlflow.log_param("model_type", "linear_regression")
        mlflow.log_param("features", X_train_lr.columns.tolist())
        mlflow.log_param("target_transform", "log1p")
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name="model")

        # Residuals plot artifact
        plot_path = f"residuals_{model_label}_lr.png"
        save_residuals_plot(y_test, y_pred, plot_path, f"{model_label} — Linear Regression")
        mlflow.log_artifact(plot_path)
        Path(plot_path).unlink()  # cleanup

        print(f"  Linear reg    | MAE: ${metrics['MAE']:>12,.0f} | "
              f"MAPE: {metrics['MAPE']:.1f}% | R²: {metrics['R2']:.3f}")
        return metrics

# ---- Run all baselines for both models ----
print("=" * 70)
print("MODEL A: Assessment-aware AVM")
print("=" * 70)
run_mean_baseline(X_train_A, X_test_A, y_train, y_test, EXPERIMENT_A, "modelA")
run_state_mean_baseline(X_train_A, X_test_A, y_train, y_test, EXPERIMENT_A, "modelA")
run_linear_regression(X_train_A, X_test_A, y_train, y_test, EXPERIMENT_A, "modelA")

print("\n" + "=" * 70)
print("MODEL B: Fundamentals-only AVM")
print("=" * 70)
run_mean_baseline(X_train_B, X_test_B, y_train, y_test, EXPERIMENT_B, "modelB")
run_state_mean_baseline(X_train_B, X_test_B, y_train, y_test, EXPERIMENT_B, "modelB")
run_linear_regression(X_train_B, X_test_B, y_train, y_test, EXPERIMENT_B, "modelB")

Tracking URI: sqlite:///mlflow.db


2026/04/26 11:47:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/26 11:47:08 INFO mlflow.store.db.utils: Updating database tables
2026/04/26 11:47:11 INFO mlflow.tracking.fluent: Experiment with name 'alloytower_avm_modelA_assessment_aware' does not exist. Creating a new experiment.
2026/04/26 11:47:11 INFO mlflow.tracking.fluent: Experiment with name 'alloytower_avm_modelB_fundamentals_only' does not exist. Creating a new experiment.


Experiment ready: alloytower_avm_modelA_assessment_aware
Experiment ready: alloytower_avm_modelB_fundamentals_only
MODEL A: Assessment-aware AVM
  Mean baseline | MAE: $     393,684 | MAPE: 67.6% | R²: -0.002
  State-mean    | MAE: $     280,003 | MAPE: 45.0% | R²: 0.517


2026/04/26 11:47:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Linear reg    | MAE: $     188,116 | MAPE: 23.0% | R²: 0.505

MODEL B: Fundamentals-only AVM
  Mean baseline | MAE: $     393,684 | MAPE: 67.6% | R²: -0.002
  State-mean    | MAE: $     280,003 | MAPE: 45.0% | R²: 0.517


2026/04/26 11:47:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Linear reg    | MAE: $     276,100 | MAPE: 39.5% | R²: 0.506


{'MAE': 276099.92345565563,
 'RMSE': np.float64(408442.2717779874),
 'R2': 0.5057661730261136,
 'MAPE': np.float64(39.50995660046438)}

In [5]:
import lightgbm as lgb

def run_lightgbm(X_train, X_test, y_train, y_test, experiment_name, model_label, features_used):
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=f"{model_label}_lightgbm"):
        # Hyperparameters tuned for small data (we have ~1500 training rows).
        params = {
            "objective": "regression",
            "metric": "mae",
            "learning_rate": 0.05,
            "num_leaves": 31,
            "max_depth": 6,
            "min_child_samples": 20,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
            "verbose": -1,
            "random_state": 42,
        }

        # log-transform target to handle right skew
        y_train_log = np.log1p(y_train)

        # Identify categorical columns for LightGBM's native handling
        cat_features = [c for c in X_train.columns if str(X_train[c].dtype) == "category"]

        train_set = lgb.Dataset(X_train, label=y_train_log, categorical_feature=cat_features)

        # Train with a fixed number of rounds (no early stopping since we're not
        # holding out a validation fold here — train/test only).
        model = lgb.train(
            params,
            train_set,
            num_boost_round=500,
        )

        y_pred_log = model.predict(X_test)
        y_pred = np.expm1(y_pred_log)
        metrics = compute_metrics(y_test, y_pred)

        # Log everything to MLflow
        mlflow.log_params(params)
        mlflow.log_param("model_type", "lightgbm")
        mlflow.log_param("num_boost_round", 500)
        mlflow.log_param("features", features_used)
        mlflow.log_param("target_transform", "log1p")
        mlflow.log_param("categorical_features", cat_features)
        mlflow.log_metrics(metrics)

        # Log model
        mlflow.lightgbm.log_model(model, name="model")

        # Feature importance artifact
        importance = pd.DataFrame({
            "feature": X_train.columns,
            "importance": model.feature_importance(importance_type="gain"),
        }).sort_values("importance", ascending=False)
        importance_path = f"feature_importance_{model_label}.csv"
        importance.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path)
        Path(importance_path).unlink()

        # Residuals plot artifact
        plot_path = f"residuals_{model_label}_lgbm.png"
        save_residuals_plot(y_test, y_pred, plot_path, f"{model_label} — LightGBM")
        mlflow.log_artifact(plot_path)
        Path(plot_path).unlink()

        print(f"  LightGBM      | MAE: ${metrics['MAE']:>12,.0f} | "
              f"MAPE: {metrics['MAPE']:.1f}% | R²: {metrics['R2']:.3f}")
        print(f"\n  Top 5 feature importances:")
        print(importance.head(5).to_string(index=False))

        return metrics, model, importance

# ---- Train LightGBM for both models ----
print("=" * 70)
print("MODEL A: Assessment-aware AVM — LightGBM")
print("=" * 70)
metrics_A, model_A, imp_A = run_lightgbm(
    X_train_A, X_test_A, y_train, y_test, EXPERIMENT_A, "modelA", MODEL_A_FEATURES
)

print("\n" + "=" * 70)
print("MODEL B: Fundamentals-only AVM — LightGBM")
print("=" * 70)
metrics_B, model_B, imp_B = run_lightgbm(
    X_train_B, X_test_B, y_train, y_test, EXPERIMENT_B, "modelB", MODEL_B_FEATURES
)

# ---- Final comparison table ----
print("\n\n" + "=" * 70)
print("FINAL COMPARISON")
print("=" * 70)

# Pull all runs from both experiments
all_runs = []
for exp_name in [EXPERIMENT_A, EXPERIMENT_B]:
    exp = mlflow.get_experiment_by_name(exp_name)
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    runs["experiment"] = exp_name
    all_runs.append(runs)

all_runs_df = pd.concat(all_runs, ignore_index=True)

# Trim to the columns we care about
summary = all_runs_df[[
    "experiment", "tags.mlflow.runName",
    "metrics.MAE", "metrics.MAPE", "metrics.R2", "metrics.RMSE",
]].copy()
summary.columns = ["Experiment", "Run", "MAE", "MAPE", "R²", "RMSE"]
summary = summary.sort_values(["Experiment", "MAE"])

print(summary.to_string(index=False))

MODEL A: Assessment-aware AVM — LightGBM
  LightGBM      | MAE: $      64,665 | MAPE: 8.3% | R²: 0.976

  Top 5 feature importances:
       feature  importance
assessed_value    4,647.71
         state      423.41
          city      389.02
          sqft      103.38
 lot_size_sqft       35.11

MODEL B: Fundamentals-only AVM — LightGBM
  LightGBM      | MAE: $     281,061 | MAPE: 41.4% | R²: 0.529

  Top 5 feature importances:
      feature  importance
        state    1,846.93
         city    1,434.66
         sqft      856.06
lot_size_sqft      604.23
   year_built      529.48


FINAL COMPARISON
                             Experiment                        Run        MAE  MAPE    R²       RMSE
 alloytower_avm_modelA_assessment_aware            modelA_lightgbm  64,665.15  8.28  0.98  90,206.62
 alloytower_avm_modelA_assessment_aware   modelA_linear_regression 188,116.40 23.02  0.51 408,717.67
 alloytower_avm_modelA_assessment_aware modelA_state_mean_baseline 280,003.45 45.03  0.52 4

In [6]:
from sklearn.model_selection import KFold
from itertools import product

# ---- Build interaction features ----
# Add a small set of interactions that *could* carry signal:
#   - state × property_type: do condos vs single-family price differently across states?
#   - sqft per bedroom: a "bigger rooms" signal that bedrooms alone misses
#   - sqft × state (target-encoded): does sqft matter more in expensive states?
#
# Keep the count small. With ~1500 training rows, every extra feature risks overfit.

def add_interactions(X):
    X = X.copy()
    # sqft per bedroom (handle div-by-zero defensively)
    X["sqft_per_bedroom"] = X["sqft"] / X["bedrooms"].replace(0, np.nan)
    X["sqft_per_bedroom"] = X["sqft_per_bedroom"].fillna(X["sqft"])
    # state × sqft (multiplicative interaction; state is already target-encoded so this works)
    X["state_x_sqft"] = X["state"] * X["sqft"]
    # building age as a feature derived from year_built (no leakage — year_built is real)
    X["building_age"] = 2024 - X["year_built"]
    return X

X_train_A_int = add_interactions(X_train_A)
X_test_A_int  = add_interactions(X_test_A)
X_train_B_int = add_interactions(X_train_B)
X_test_B_int  = add_interactions(X_test_B)

print(f"Model A features (with interactions): {X_train_A_int.shape[1]}")
print(f"Model B features (with interactions): {X_train_B_int.shape[1]}")

# ---- Hyperparameter grid ----
# Small, focused grid. 9 combinations × 5 folds = 45 fits per model. Fast on this size.
PARAM_GRID = {
    "num_leaves": [15, 31, 63],
    "min_child_samples": [10, 20, 40],
}

BASE_PARAMS = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "max_depth": 6,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 42,
}
NUM_BOOST_ROUND = 500
N_FOLDS = 5

# ---- Cross-validated tuning + evaluation ----
def cv_tune_and_evaluate(X_train, X_test, y_train, y_test, model_label, experiment_name):
    """Grid-search over PARAM_GRID with CV, then refit best on full train and score on test."""
    cat_features = [c for c in X_train.columns if str(X_train[c].dtype) == "category"]

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    # Step 1: find best params via CV on training set
    best_score = float("inf")
    best_params = None
    grid_results = []

    for num_leaves, min_child in product(PARAM_GRID["num_leaves"], PARAM_GRID["min_child_samples"]):
        params = {**BASE_PARAMS, "num_leaves": num_leaves, "min_child_samples": min_child}
        fold_maes = []

        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            y_tr_log = np.log1p(y_tr)

            dtrain = lgb.Dataset(X_tr, label=y_tr_log, categorical_feature=cat_features)
            model = lgb.train(params, dtrain, num_boost_round=NUM_BOOST_ROUND)
            y_val_pred = np.expm1(model.predict(X_val))
            fold_maes.append(mean_absolute_error(y_val, y_val_pred))

        mean_mae = np.mean(fold_maes)
        std_mae = np.std(fold_maes)
        grid_results.append({
            "num_leaves": num_leaves,
            "min_child_samples": min_child,
            "cv_mae_mean": mean_mae,
            "cv_mae_std": std_mae,
        })
        if mean_mae < best_score:
            best_score = mean_mae
            best_params = params

    grid_df = pd.DataFrame(grid_results).sort_values("cv_mae_mean")
    print(f"\n  Top 3 hyperparameter combos for {model_label} (by CV MAE):")
    print(grid_df.head(3).to_string(index=False))

    # Step 2: 5-fold CV with best params to get confidence intervals on headline metrics
    fold_metrics = []
    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        y_tr_log = np.log1p(y_tr)

        dtrain = lgb.Dataset(X_tr, label=y_tr_log, categorical_feature=cat_features)
        model = lgb.train(best_params, dtrain, num_boost_round=NUM_BOOST_ROUND)
        y_val_pred = np.expm1(model.predict(X_val))
        fold_metrics.append(compute_metrics(y_val, y_val_pred))

    cv_summary = pd.DataFrame(fold_metrics).agg(["mean", "std"]).round(4)

    # Step 3: refit best params on full training set, score on held-out test
    y_train_log = np.log1p(y_train)
    final_dtrain = lgb.Dataset(X_train, label=y_train_log, categorical_feature=cat_features)
    final_model = lgb.train(best_params, final_dtrain, num_boost_round=NUM_BOOST_ROUND)
    y_test_pred = np.expm1(final_model.predict(X_test))
    test_metrics = compute_metrics(y_test, y_test_pred)

    # Log to MLflow
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=f"{model_label}_lightgbm_tuned_cv"):
        mlflow.log_params(best_params)
        mlflow.log_param("model_type", "lightgbm_tuned")
        mlflow.log_param("num_features", X_train.shape[1])
        mlflow.log_param("interactions_added", True)
        mlflow.log_param("cv_folds", N_FOLDS)

        # Test-set metrics (the headline)
        mlflow.log_metrics(test_metrics)

        # CV mean ± std for each metric
        for metric in ["MAE", "MAPE", "R2", "RMSE"]:
            mlflow.log_metric(f"cv_{metric}_mean", cv_summary.loc["mean", metric])
            mlflow.log_metric(f"cv_{metric}_std", cv_summary.loc["std", metric])

        # Feature importance
        importance = pd.DataFrame({
            "feature": X_train.columns,
            "importance": final_model.feature_importance(importance_type="gain"),
        }).sort_values("importance", ascending=False)
        importance_path = f"feature_importance_{model_label}_tuned.csv"
        importance.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path)
        Path(importance_path).unlink()

        mlflow.lightgbm.log_model(final_model, name="model")

    return test_metrics, cv_summary, grid_df, importance, final_model

# ---- Run for both models ----
print("=" * 70)
print("MODEL A — TUNED + CV")
print("=" * 70)
test_A, cv_A, grid_A, imp_A_tuned, model_A_tuned = cv_tune_and_evaluate(
    X_train_A_int, X_test_A_int, y_train, y_test, "modelA", EXPERIMENT_A
)

print(f"\n  Test set: MAE ${test_A['MAE']:,.0f} | MAPE {test_A['MAPE']:.1f}% | R² {test_A['R2']:.3f}")
print(f"  CV mean : MAE ${cv_A.loc['mean','MAE']:,.0f} ± ${cv_A.loc['std','MAE']:,.0f}")
print(f"  CV mean : MAPE {cv_A.loc['mean','MAPE']:.1f}% ± {cv_A.loc['std','MAPE']:.1f}%")
print(f"  CV mean : R² {cv_A.loc['mean','R2']:.3f} ± {cv_A.loc['std','R2']:.3f}")
print(f"\n  Top 5 features:")
print(imp_A_tuned.head(5).to_string(index=False))

print("\n" + "=" * 70)
print("MODEL B — TUNED + CV")
print("=" * 70)
test_B, cv_B, grid_B, imp_B_tuned, model_B_tuned = cv_tune_and_evaluate(
    X_train_B_int, X_test_B_int, y_train, y_test, "modelB", EXPERIMENT_B
)

print(f"\n  Test set: MAE ${test_B['MAE']:,.0f} | MAPE {test_B['MAPE']:.1f}% | R² {test_B['R2']:.3f}")
print(f"  CV mean : MAE ${cv_B.loc['mean','MAE']:,.0f} ± ${cv_B.loc['std','MAE']:,.0f}")
print(f"  CV mean : MAPE {cv_B.loc['mean','MAPE']:.1f}% ± {cv_B.loc['std','MAPE']:.1f}%")
print(f"  CV mean : R² {cv_B.loc['mean','R2']:.3f} ± {cv_B.loc['std','R2']:.3f}")
print(f"\n  Top 5 features:")
print(imp_B_tuned.head(5).to_string(index=False))

# ---- Final lift summary ----
print("\n\n" + "=" * 70)
print("LIFT FROM TUNING + INTERACTIONS")
print("=" * 70)

# Pull baseline LightGBM metrics from earlier (you have these from Cell 3 output)
# Compare the tuned model to the basic Cell 3 LightGBM
print(f"\nModel A:")
print(f"  Cell 3 LightGBM      : MAE $64,665 | MAPE 8.3%  | R² 0.976")
print(f"  Cell 4 LightGBM tuned: MAE ${test_A['MAE']:,.0f} | MAPE {test_A['MAPE']:.1f}% | R² {test_A['R2']:.3f}")

print(f"\nModel B:")
print(f"  Cell 3 LightGBM      : MAE $281,061 | MAPE 41.4% | R² 0.529")
print(f"  Cell 4 LightGBM tuned: MAE ${test_B['MAE']:,.0f} | MAPE {test_B['MAPE']:.1f}% | R² {test_B['R2']:.3f}")

Model A features (with interactions): 14
Model B features (with interactions): 13
MODEL A — TUNED + CV

  Top 3 hyperparameter combos for modelA (by CV MAE):
 num_leaves  min_child_samples  cv_mae_mean  cv_mae_std
         15                 10    63,712.70    3,183.04
         15                 20    63,806.52    3,401.17
         63                 20    64,431.56    2,179.74

  Test set: MAE $62,196 | MAPE 8.0% | R² 0.978
  CV mean : MAE $63,713 ± $3,559
  CV mean : MAPE 8.0% ± 0.3%
  CV mean : R² 0.980 ± 0.003

  Top 5 features:
       feature  importance
assessed_value    5,021.08
         state      401.79
          city       90.55
  state_x_sqft       49.04
          sqft       30.22

MODEL B — TUNED + CV

  Top 3 hyperparameter combos for modelB (by CV MAE):
 num_leaves  min_child_samples  cv_mae_mean  cv_mae_std
         31                 40   309,465.91   12,237.83
         63                 40   309,465.91   12,237.83
         15                 40   310,016.24   12,249.